# FaithfulRecon: RQ1 pilot (lighting inconsistency → floaters)

Each of the 8 training views gets its own camera-style change of exposure, white balance and gamma, applied inside the object mask.
Held-out views keep their original appearance.
We measure whether floaters increase with the level `none → mild → strong`:
- 3 scans: 24, 63, 83
- 3 seeds each
- training views: 8

**Before running:**
1. Runtime → Change runtime type → **GPU** (A100/L4 preferred; T4 works but is slower).
2. Colab **Secrets** (key icon): add `GH_TOKEN`, a fine-grained GitHub token with **read-only** access to the repo. Do not paste the token into this notebook.
3. Fill in the data links in the *Data* section.

Runs, data and results are stored on Drive, so if the session disconnects you just re-run the cells; finished steps are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
import os, subprocess, pathlib, shutil
from google.colab import drive, userdata
drive.mount('/content/drive')

STORE = '/content/drive/MyDrive/faithfulrecon_store'   # persistent: runs/, data/, results/
REPO_URL = 'github.com/Huyrua27/faithfulrecon.git'
BRANCH = 'feature/huy-dtu-multiscan'                    # branch with photometric.py + floater_metrics.py
REPO = '/content/faithfulrecon'

for d in ('runs', 'data', 'results'):
    os.makedirs(f'{STORE}/{d}', exist_ok=True)
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', f'https://{token}@{REPO_URL}', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)  # no token on disk
del token
for d in ('runs', 'data'):                               # runs/ and data/ -> Drive
    p = pathlib.Path(REPO, d)
    if p.is_symlink():
        p.unlink()
    elif p.exists():
        shutil.rmtree(p)
    p.symlink_to(f'{STORE}/{d}')
%cd {REPO}
!git log --oneline -1
assert os.path.exists('frecon/data/photometric.py'), 'Wrong branch: photometric.py is missing (has it been pushed?)'

In [ ]:
!pip -q install -r requirements.txt
!python -m pytest -q tests/test_photometric.py tests/test_dtu.py

## 2. DTU data (3 scans)

Two sources are needed:
- **Preprocessed DTU in the IDR/NeuS format**, with `image/`, `mask/` and `cameras_sphere.npz` for each scan.
  Use the link from the NeuS (`data_DTU.zip`) or 2DGS README, the same archive used before.
  Either a direct URL or a Google Drive link works (handled by `gdown`).
- **STL reference geometry**: the official *Points* archive of DTU MVS 2014, https://roboimagedata.compute.dtu.dk/?page_id=36, about 6.3 GB.
  It is only downloaded once. After the `ref_points.npy` cache files are built, the archive can be deleted.

Fallback: if you already have a zip of `data/dtu/scanXX/...`, upload it to Drive and set `DTU_PREPROC_ZIP_ON_DRIVE`.

In [ ]:
SCANS = [24, 63, 83]           # many / medium / few floaters under consistent lighting (51% / 22% / 7%)
DTU_PREPROC_URL = ''           # URL or Google Drive link of the preprocessed DTU zip (IDR/NeuS)
DTU_PREPROC_ZIP_ON_DRIVE = ''  # or: path of a zip already on Drive, e.g. f'{STORE}/data_DTU.zip'
DTU_POINTS_URL = ''            # official Points.zip URL (from the DTU page above)

import glob, zipfile
os.makedirs('data/dtu/Points/stl', exist_ok=True)
need_scan = [s for s in SCANS if not os.path.exists(f'data/dtu/scan{s}/cameras_sphere.npz')
             and not os.path.exists(f'data/dtu/scan{s}/cameras.npz')]
if need_scan:
    src = DTU_PREPROC_ZIP_ON_DRIVE or '/content/dtu_preproc.zip'
    if not DTU_PREPROC_ZIP_ON_DRIVE:
        assert DTU_PREPROC_URL, f'Missing data for scans {need_scan}: fill in DTU_PREPROC_URL'
        if 'drive.google.com' in DTU_PREPROC_URL:
            !pip -q install gdown && gdown --fuzzy "{DTU_PREPROC_URL}" -O {src}
        else:
            !wget -q -O {src} "{DTU_PREPROC_URL}"
    !unzip -q -o "{src}" -d /content/dtu_preproc
    for s in need_scan:          # folder names differ between releases: scan24, dtu_scan24, ...
        hits = [os.path.dirname(p) for p in glob.glob('/content/dtu_preproc/**/cameras*.npz', recursive=True)
                if os.path.basename(os.path.dirname(p)).endswith(f'scan{s}')]
        assert hits, f'scan{s} not found in the archive'
        shutil.copytree(hits[0], f'data/dtu/scan{s}', dirs_exist_ok=True)

need_stl = [s for s in SCANS if not os.path.exists(f'data/dtu/Points/stl/stl{s:03d}_total.ply')
            and not os.path.exists(f'data/dtu/scan{s}/ref_points.npy')]
if need_stl:
    assert DTU_POINTS_URL, f'Missing STL for scans {need_stl}: fill in DTU_POINTS_URL'
    if not os.path.exists('/content/dtu_points.zip'):
        !wget -q --show-progress -O /content/dtu_points.zip "{DTU_POINTS_URL}"
    with zipfile.ZipFile('/content/dtu_points.zip') as z:
        for s in need_stl:
            name = next(n for n in z.namelist() if n.endswith(f'stl{s:03d}_total.ply'))
            with z.open(name) as f_in, open(f'data/dtu/Points/stl/stl{s:03d}_total.ply', 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

# build the reference-point cache once (avoids parallel workers racing on the same file)
from frecon.data.dtu import load_dtu
for s in SCANS:
    cache = f'data/dtu/scan{s}/ref_points.npy'
    if not os.path.exists(cache):
        _, _, ref, _ = load_dtu(f'data/dtu/scan{s}', 8, 16, 4, stl_ply=f'data/dtu/Points/stl/stl{s:03d}_total.ply',
                                ref_cache=cache)
        print(f'scan{s}: {len(ref)} reference points')
    print(f'scan{s}:', sorted(os.listdir(f'data/dtu/scan{s}')))

## 3. Configs: 3 levels × 3 scans

The `none` level is retrained with the current code, so all three levels are directly comparable.

In [ ]:
import yaml
SEEDS = [0, 1, 2]
LEVELS = {
    'none':   None,
    'mild':   dict(exposure_stops=0.5, wb=0.05, gamma=0.10, seed=0),
    'strong': dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=0),
}
template = pathlib.Path('configs/dtu_scan24_v8.yaml').read_text(encoding='utf-8')
CFG = {}
for s in SCANS:
    base = yaml.safe_load(template.replace('scan24', f'scan{s}').replace('stl024', f'stl{s:03d}'))
    for name, p in LEVELS.items():
        cfg = dict(base, out_dir=f'runs/photo_{name}_dtu_scan{s}_v8', seeds=SEEDS)
        cfg['scene'] = dict(base['scene'])
        if p:
            cfg['scene']['photometric'] = p
        path = f'configs/photo_{name}_dtu_scan{s}_v8.yaml'
        with open(path, 'w', encoding='utf-8') as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        CFG[(s, name)] = path
for k, v in CFG.items(): print(k, v)

## 4. Sanity check: perturbed training views (scan24, strong level)

In [ ]:
import matplotlib.pyplot as plt
sc = yaml.safe_load(open(CFG[(24, 'strong')], encoding='utf-8'))['scene']
tr0, *_ = load_dtu(sc['root'], sc['n_train_views'], sc['n_test_views'], sc['downscale'])
tr1, _, _, info = load_dtu(sc['root'], sc['n_train_views'], sc['n_test_views'], sc['downscale'],
                           photometric=sc['photometric'])
fig, ax = plt.subplots(2, 8, figsize=(20, 4.2))
for k in range(8):
    ax[0, k].imshow(tr0[k].image.cpu().numpy()); ax[1, k].imshow(tr1[k].image.cpu().numpy())
    p = info['photometric'][k]
    ax[1, k].set_title(f"gain={p['gain']:.2f}  γ={p['gamma']:.2f}", fontsize=9)
for a in ax.flat: a.axis('off')
ax[0, 0].set_title('original', fontsize=9, loc='left')
plt.tight_layout(); plt.show()

## 5. Train the base models (27 runs)

Each scan runs its levels × seeds in parallel.
- **≥ 35 GB GPU** (A100): all 9 runs at once.
- **Smaller GPU** (T4/L4): one level at a time, 3 runs.

Rough time on an A100: ~35 min for scan24 and ~10–15 min each for scans 63 and 83.
If the session disconnects, re-run this cell: seeds that already have `base.pt` are skipped.

In [ ]:
import sys, time, torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
ALL_LEVELS_AT_ONCE = gpu_gb >= 35
print(f'GPU {gpu_gb:.0f} GB -> ' + ('all levels in parallel' if ALL_LEVELS_AT_ONCE else 'one level at a time'))

def launch(cfg):
    return subprocess.Popen([sys.executable, 'scripts/run_parallel.py', '--script', 'scripts/phase0.py',
                             '--config', cfg, '--workers', str(len(SEEDS)), '--base_only'])

for s in SCANS:
    t0 = time.time()
    todo = [n for n in LEVELS
            if not all(os.path.exists(f'runs/photo_{n}_dtu_scan{s}_v8/seed_{k}/base.pt') for k in SEEDS)]
    if ALL_LEVELS_AT_ONCE:
        codes = [p.wait() for p in [launch(CFG[(s, n)]) for n in todo]]
    else:
        codes = [launch(CFG[(s, n)]).wait() for n in todo]
    print(f'scan{s}: levels {todo} -> exit {codes} ({(time.time() - t0) / 60:.1f} min)')
    if any(codes):
        print(f'ERROR: see runs/photo_*_dtu_scan{s}_v8/parallel_w*.log')

## 6. Floater metrics and summary

| Column | Meaning |
|---|---|
| `off_01` | share of Gaussians farther than 0.1 from the reference surface |
| `wfloat_01` | share of held-out blending weight from Gaussians with d > 0.1, i.e. floaters that actually show up. **Main metric** |
| `front_01` | share of floaters lying in front of the surface |
| `acc` / `comp` | accuracy / completeness (lower is better), Gaussians with opacity > 0.05 |
| `psnr` | PSNR against the **original** held-out images. It drops naturally because the colours were shifted, so **it is not evidence about floaters** |

In [ ]:
import pandas as pd, numpy as np
os.makedirs('results/photometric', exist_ok=True)
dfs = []
for s in SCANS:
    runs = ' '.join(f'runs/photo_{n}_dtu_scan{s}_v8' for n in LEVELS)
    out = f'results/photometric/floater_metrics_scan{s}.csv'
    !python -W ignore scripts/floater_metrics.py --run_dirs {runs} --seeds {' '.join(map(str, SEEDS))} --out {out}
    dfs.append(pd.read_csv(out))
df = pd.concat(dfs, ignore_index=True)
df['level'] = df.run.astype(str).str.extract(r'photo_(\w+?)_dtu')[0]
order = list(LEVELS)
df.to_csv('results/photometric/floater_metrics_all.csv', index=False)

M = ['off_01', 'front_01', 'wfloat_01', 'acc', 'comp', 'psnr', 'n_gaussians']
summ = df.groupby(['scan', 'level'])[M].agg(['mean', 'std']).reindex(order, level='level')
print(summ.round(4).to_string())

base = df[df.level == 'none'].set_index(['scan', 'seed'])
rows = []
for n in order[1:]:
    cur = df[df.level == n].set_index(['scan', 'seed'])
    d = (cur[M] - base[M]).dropna()
    rows.append({'level': n, 'pairs': len(d),
                 **{f'Δ{m}': f"{d[m].mean():+.4f} ({int((d[m] > 0).sum())}/{len(d)} up)" for m in M}})
delta = pd.DataFrame(rows)
print('\n== Change vs consistent lighting (paired: same scan, same seed) ==')
print(delta.to_string(index=False))
delta.to_csv('results/photometric/delta_vs_none.csv', index=False)

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(20, 3.6), layout='constrained')
for ax, (m, lab) in zip(axs, [('off_01', 'Gaussians with d > 0.1'), ('wfloat_01', 'render weight of floaters'),
                              ('front_01', 'floaters in front of surface'), ('comp', 'completeness (lower = better)'),
                              ('psnr', 'held-out PSNR')]):
    for s in SCANS:
        g = df[df.scan.astype(str).str.endswith(str(s))].groupby('level')[m].agg(['mean', 'std']).reindex(order)
        ax.errorbar(range(len(order)), g['mean'], yerr=g['std'], marker='o', capsize=3, label=f'scan{s}')
    ax.set_xticks(range(len(order)), order); ax.set_title(lab, fontsize=10)
axs[0].legend(fontsize=8)
fig.savefig('results/photometric/dose_response.png', dpi=150); plt.show()
!mkdir -p {STORE}/results/photometric && cp results/photometric/* {STORE}/results/photometric/
!ls {STORE}/results/photometric